# Multi-Document Analysis with ragged

**Purpose:** Advanced techniques for working with multiple documents

**Topics Covered:**
- Batch PDF ingestion
- Cross-document search
- Metadata filtering
- Comparative analysis
- Document versioning

**Prerequisites:**
- Completed `01-getting-started.ipynb`
- Multiple sample PDFs available
- ChromaDB and Ollama running

---

## 1. Batch PDF Ingestion

Ingest multiple documents efficiently in a single operation.

In [ ]:
# Clear existing documents for clean demo
!ragged clear --force

In [ ]:
# Ingest all PDFs in the sample_documents directory
!ragged ingest pdf ../sample_documents/*.pdf

In [ ]:
# Verify all documents were ingested
!ragged list

## 2. Cross-Document Search

Search across all ingested documents to find relevant information.

In [ ]:
# Search for machine learning concepts across all documents
!ragged query text "machine learning algorithms" --limit 5

In [ ]:
# Search with higher relevance threshold
!ragged query text "neural network architecture" --limit 5 --min-score 0.75

## 3. Metadata-Based Filtering

Use metadata to narrow searches to specific documents or categories.

In [ ]:
# Search within a specific document using path filter
!ragged search --path "data_visualization.pdf" --metadata "category=charts"

In [ ]:
# Search with multiple metadata filters
!ragged search --metadata "category=research" --metadata "priority=high"

## 4. Programmatic Multi-Document Analysis

Use Python API for advanced document comparison and analysis.

In [ ]:
from ragged.storage.vector_store import VectorStore
from ragged.retrieval.retriever import Retriever
from collections import defaultdict

# Initialize components
store = VectorStore()
retriever = Retriever()

# Get all documents
all_docs = store.get_all_documents()
print(f"Total documents: {len(all_docs)}")

In [ ]:
# Compare document sizes and chunk counts
doc_stats = defaultdict(dict)

for doc in all_docs:
    doc_path = doc.get('document_path', 'unknown')
    doc_stats[doc_path]['chunk_count'] = doc_stats[doc_path].get('chunk_count', 0) + 1

# Display statistics
print("\nDocument Statistics:")
print("-" * 60)
for path, stats in sorted(doc_stats.items()):
    print(f"{path}: {stats['chunk_count']} chunks")

In [ ]:
# Find which documents contain specific concepts
query = "data visualization techniques"
results = retriever.retrieve(query, k=10)

# Group results by document
docs_with_concept = defaultdict(list)
for result in results:
    doc_path = result.get('metadata', {}).get('document_path', 'unknown')
    score = result.get('score', 0)
    docs_with_concept[doc_path].append(score)

# Display relevance by document
print(f"\nDocuments containing '{query}':")
print("-" * 60)
for doc, scores in sorted(docs_with_concept.items(), 
                          key=lambda x: max(x[1]), reverse=True):
    avg_score = sum(scores) / len(scores)
    max_score = max(scores)
    print(f"{doc}:")
    print(f"  Matches: {len(scores)}")
    print(f"  Max relevance: {max_score:.3f}")
    print(f"  Avg relevance: {avg_score:.3f}")
    print()

## 5. Comparative Document Analysis

Compare content and themes across multiple documents.

In [ ]:
# Define topics to compare across documents
topics = [
    "machine learning",
    "data visualization",
    "neural networks",
    "statistical analysis"
]

# Analyse topic coverage across documents
topic_coverage = {}

for topic in topics:
    results = retriever.retrieve(topic, k=5)
    docs = set(r.get('metadata', {}).get('document_path', 'unknown') 
               for r in results)
    topic_coverage[topic] = docs

# Display topic coverage matrix
print("Topic Coverage Matrix:")
print("=" * 80)
for topic, docs in topic_coverage.items():
    print(f"\n{topic.upper()}:")
    for doc in sorted(docs):
        print(f"  ✓ {doc}")

## 6. Document Metadata Management

Add and query custom metadata for better organization.

In [ ]:
# View current metadata
!ragged metadata list

In [ ]:
# Add custom metadata (if command available)
# !ragged metadata add --document-id <id> --key "category" --value "research"
# !ragged metadata add --document-id <id> --key "priority" --value "high"

## 7. Storage Management for Multi-Document Collections

Manage storage efficiently with large document collections.

In [ ]:
# View detailed storage information
!ragged storage info --format json

In [ ]:
# Optimize storage (vacuum)
!ragged storage vacuum

In [ ]:
# Export data for backup
!ragged export backup --output ./backup.json

## 8. Advanced Query Techniques

Combine filters and ranking for precise retrieval.

In [ ]:
# Hybrid search with metadata filtering
!ragged search "machine learning algorithms" \
    --metadata "category=research" \
    --min-score 0.7 \
    --limit 5

In [ ]:
# Show content preview with results
!ragged search "neural network architecture" \
    --show-content \
    --limit 3

## 9. Batch Processing Best Practices

Tips for efficient multi-document workflows.

In [ ]:
# Process documents with progress tracking
import os
from pathlib import Path

# Get all PDFs
pdf_dir = Path('../sample_documents')
pdfs = list(pdf_dir.glob('*.pdf'))

print(f"Found {len(pdfs)} PDF files:")
for pdf in pdfs:
    print(f"  - {pdf.name} ({pdf.stat().st_size / 1024:.1f} KB)")

## 10. Next Steps

**Explore further:**
1. Add vision embeddings for multi-modal search
2. Use templates for repeatable queries
3. Implement document versioning
4. Set up automated monitoring

**Related notebooks:**
- `03-gpu-optimization.ipynb` - Performance tuning

**Documentation:**
- [Multi-Modal Workflow Tutorial](../../docs/tutorials/multimodal-workflow.md)
- [Advanced Search Guide](../../docs/guides/advanced-search.md)
- [Storage Management](../../docs/guides/storage-management.md)

---